### Homework 04 - Measurements of rectangular tables
Igor Di Tota

In this exercise, I will study how the tolerances on the dimensions of a rectangular table propagate to quantities that depend on its dimensions.

The nominal dimensions of the table are length $L = 160$ cm and width $W = 90$ cm, with standard deviations $\sigma_L = 0.5$ cm and $\sigma_W = 0.3$ cm.
Length and width are assumed to be two independent random variables, and the table sides are always perpendicular.

The quantities we want to study are:
1.  table surface, $S = L \, W$
2.  total length of the table sides (perimeter), $P = 2(L + W)$
3.  table diagonal, $D = \sqrt{L^2 + W^2}$

For each of them the expected standard deviation will be computed, toghether with the correlation between $S$ and $P$.
Two ways will be used:
- **analytically**, using the error propagation formula;
- **numerically**, generating many tables with $L$ and $W$ drawn from Gaussian distributions.

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import os
os.makedirs("images", exist_ok=True)

rng = np.random.default_rng(23)

# Nominal dimensions in cm
L = 160
W = 90

# Production tolerances in cm (standard deviations)
sigma_L = 0.5
sigma_W = 0.3

### Analytical error propagation

The input variables are $\mathbf{X} = (L, W)$.

Since length and width are independent, their covariance matrix is written in diagonal form as
$$
\mathbb{C}_X =
\begin{pmatrix}
\sigma_L^2 & 0 \\
0 & \sigma_W^2
\end{pmatrix}.
$$

The output variables are $\mathbf{Y} = (S, P, D)$, with
$$
S = L\, W, \qquad P = 2(L + W), \qquad D = \sqrt{L^2 + W^2}.
$$

To propagate the uncertainties from $\mathbf{X}$ to $\mathbf{Y}$, we use the general error propagation formula seen in the lecture:
$$
\mathbb{C}_Y = A\, \mathbb{C}_X\, A^\intercal ,
$$
where $A$ is the matrix of partial derivatives $A_{ij} = \partial y_i / \partial x_j$ for transformation to $\mathbf{Y}$, evaluated at the nominal values $L = 160$ cm and $W = 90$ cm:
$$
A =
\begin{pmatrix}
\dfrac{\partial S}{\partial L} & \dfrac{\partial S}{\partial W} \\[2ex]
\dfrac{\partial P}{\partial L} & \dfrac{\partial P}{\partial W} \\[2ex]
\dfrac{\partial D}{\partial L} & \dfrac{\partial D}{\partial W}
\end{pmatrix}
=
\begin{pmatrix}
W & L \\[1ex]
2 & 2 \\[1ex]
\dfrac{L}{D} & \dfrac{W}{D}
\end{pmatrix}.
$$

This formula comes from a first order Taylor expansion of $\mathbf{Y}(\mathbf{X})$ around the nominal values, so it is only an approximation for non-linear functions such as $S$ and $D$.
It is exact for the perimeter $P$, which is linear in $L$ and $W$.

The approximation is good when the standard deviations are small compared to the nominal values, as in our case ($\sigma_L / L \approx 0.3\%$, $\sigma_W / W \approx 0.3\%$).

The elements we are interested in are
$$
\sigma_S^2 = W^2\sigma_L^2 + L^2\sigma_W^2, \qquad
\sigma_P^2 = 4\left(\sigma_L^2 + \sigma_W^2\right), \qquad
\sigma_D^2 = \frac{L^2\sigma_L^2 + W^2\sigma_W^2}{L^2 + W^2},
$$
$$
\mathrm{Cov}(S, P) = 2\left(W\,\sigma_L^2 + L\,\sigma_W^2\right).
$$

Finally, the correlation coefficient between the surface and the total edge length is
$$
\rho(S, P) = \frac{\mathrm{Cov}(S, P)}{\sigma_S\, \sigma_P}.
$$


In [3]:
# Nominal diagonal
D = np.sqrt(L**2 + W**2)

# Covariance matrix of the input variables (L, W)
C_X = np.array([[sigma_L**2, 0],
                [0, sigma_W**2]])

# Matrix of partial derivatives, rows: S, P, D, columns: L, W
A = np.array([[W,     L    ],
              [2,     2    ],
              [L / D, W / D]])

# Covariance matrix of the output variables (S, P, D)
C_Y = A @ C_X @ A.T

print("C_Y =")
print(C_Y)


C_Y =
[[4.32900000e+03 7.38000000e+01 2.66702114e+01]
 [7.38000000e+01 1.36000000e+00 5.24034792e-01]
 [2.66702114e+01 5.24034792e-01 2.11543027e-01]]


In [7]:
# Standard deviations from the diagonal of C_Y
sigma_S, sigma_P, sigma_D = np.sqrt(np.diag(C_Y))

# Covariance and correlation between S and P
cov_SP = C_Y[0, 1]
rho_SP = cov_SP / (sigma_S * sigma_P)

print(f"S = {L * W:.1f} ± {sigma_S:.1f} cm^2     (relative: {sigma_S / (L * W) * 100:.2f}%)")
print(f"P = {2 * (L + W):.2f} ± {sigma_P:.2f} cm       (relative: {sigma_P / (2 * (L + W)) * 100:.2f}%)")
print(f"D = {D:.2f} ± {sigma_D:.2f} cm       (relative: {sigma_D / D * 100:.2f}%)")
print(f"\nCov(S, P) = {cov_SP:.2f} cm^3")
print(f"rho(S, P) = {rho_SP:.4f}")



S = 14400.0 ± 65.8 cm^2     (relative: 0.46%)
P = 500.00 ± 1.17 cm       (relative: 0.23%)
D = 183.58 ± 0.46 cm       (relative: 0.25%)

Cov(S, P) = 73.80 cm^3
rho(S, P) = 0.9618
